In [2]:
from pathlib import Path
import os, json, time
import duckdb
import numpy as np
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from sentence_transformers import SentenceTransformer

DATA = Path("data")
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

arts = con.execute(f"""
    SELECT article_id, prod_name, product_type_name, colour_group_name,
           graphical_appearance_name, index_group_name, department_name,
           COALESCE(detail_desc, '') AS detail_desc
    FROM '{(DATA / 'articles.parquet').as_posix()}' ORDER BY article_id
""").df()
emb = np.load(DATA / "article_emb.npy")
emb_ids = np.load(DATA / "article_emb_ids.npy")
assert (arts["article_id"].to_numpy() == emb_ids).all(), "Row order mismatch"

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

load_dotenv()
print("Keys found in .env:", list(dotenv_values().keys()))   # names only, never values
print(f"Articles: {len(arts):,} | embeddings: {emb.shape}")

Keys found in .env: ['GEMINI_API_KEY']
Articles: 105,542 | embeddings: (105542, 384)


In [3]:
available = con.execute(f"""
    SELECT DISTINCT article_id FROM '{(DATA / 'transactions_train.parquet').as_posix()}'
    WHERE t_dat > DATE '2020-09-22' - INTERVAL 56 DAY
""").df()["article_id"]
mask = arts["article_id"].isin(set(available)).to_numpy()
cat = arts[mask].reset_index(drop=True)
cat_emb = emb[mask]
cat["product_code"] = cat["article_id"].str[:7]
print(f"Searchable catalog: {len(cat):,} items currently selling")

def retrieve(query, k=10):
    q = model.encode([query], normalize_embeddings=True)[0]
    scores = cat_emb @ q
    order = np.argsort(-scores)
    out, seen = [], set()
    for i in order:
        pc = cat.at[i, "product_code"]
        if pc in seen:
            continue
        seen.add(pc)
        r = cat.iloc[i]
        out.append({"article_id": r["article_id"], "name": r["prod_name"],
                    "type": r["product_type_name"], "colour": r["colour_group_name"],
                    "section": r["index_group_name"], "desc": r["detail_desc"][:200],
                    "score": round(float(scores[i]), 3)})
        if len(out) == k:
            break
    return out

for q in ["black dress for a summer wedding", "warm winter jacket for men", "kids pyjamas with animal print"]:
    print(f"\n>>> {q}")
    for r in retrieve(q, k=5):
        print(f"   {r['score']}  {r['name']} | {r['type']} | {r['colour']} | {r['section']}")

Searchable catalog: 35,576 items currently selling

>>> black dress for a summer wedding
   0.587  Summer nights dress | Dress | Black | Ladieswear
   0.583  Summer dress | Dress | Black | Ladieswear
   0.556  Best Day Wedding Top | Top | Off White | Ladieswear
   0.553  Summer strap dress | Dress | Black | Divided
   0.552  Bridesmaid | Dress | Light Blue | Ladieswear

>>> warm winter jacket for men
   0.558  Go light jacket | Jacket | Dark Green | Sport
   0.549  Snow short jacket | Jacket | Black | Baby/Children
   0.536  Andreas | Jacket | Black | Menswear
   0.533  Dragomir Jacket | Jacket | Black | Sport
   0.531  Rocket hooded jacket | Jacket | Dark Orange | Menswear

>>> kids pyjamas with animal print
   0.571  PETAL PJ | Pyjama set | White | Baby/Children
   0.567  Leon l/l pj SB | Pyjama set | Dark Blue | Baby/Children
   0.567  TILDA PJ | Pyjama set | Light Green | Baby/Children
   0.566  4PCS LS PJ Girly with Frill | Pyjama set | Light Red | Baby/Children
   0.559  Low Pric

In [4]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"),
                      http_options=types.HttpOptions(timeout=60_000))   # 60 s per request
MODELS = ["gemini-flash-latest", "gemini-flash-lite-latest"]           # backup if the first is overloaded

SYSTEM = """You are a shopping assistant for H&M.
You may ONLY recommend products from the PRODUCTS list you are given.
Never invent products, names, colours, prices or details that are not in the list.
Pick up to 5 products that best match the request.
Leave out products that only partly match (wrong colour, wrong type, wrong section such as kids vs men).
If nothing in the list matches, say so honestly and recommend nothing.
Reply as JSON with two keys:
"answer": 2 to 4 short, friendly sentences for the customer, naming the products you picked.
"article_ids": the article_id values you recommend, copied exactly from the list."""

def call_gemini(prompt, retries=6):
    wait = 5
    for attempt in range(1, retries + 1):
        model_name = MODELS[(attempt - 1) % len(MODELS)]
        try:
            resp = client.models.generate_content(
                model=model_name, contents=prompt,
                config=types.GenerateContentConfig(system_instruction=SYSTEM,
                                                   response_mime_type="application/json",
                                                   temperature=0))
            return resp, model_name
        except Exception as e:
            if attempt == retries:
                raise
            print(f"  attempt {attempt} with {model_name} failed ({type(e).__name__}). Waiting {wait}s...")
            time.sleep(wait)
            wait = min(wait * 2, 40)

def ask(query, k=20):
    items = retrieve(query, k)
    shown = [{key: r[key] for key in ["article_id", "name", "type", "colour", "section", "desc"]} for r in items]
    prompt = f"REQUEST: {query}\n\nPRODUCTS:\n{json.dumps(shown, ensure_ascii=False)}"
    resp, used = call_gemini(prompt)
    return items, json.loads(resp.text), used

t0 = time.time()
items, out, used = ask("warm winter jacket for men")
print(f"Model used: {used} | {time.time() - t0:.1f}s\n")
print(out["answer"], "\n")
names = {r["article_id"]: f"{r['name']} | {r['type']} | {r['colour']} | {r['section']}" for r in items}
for a in out["article_ids"]:
    print(" ", a, "->", names.get(a, "NOT IN RETRIEVED LIST"))

  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
Model used: gemini-flash-lite-latest | 26.1s

I found some great padded and down jackets for men that will help keep you warm this winter, including the Peter downfree jacket, the Uno lightweight down jacket, and the Mr Tom padded jacket. Take a look and stay cozy! 

  0778645001 -> Peter downfree jacket | Unknown | Black | Menswear
  0899020002 -> Uno | Jacket | Dark Blue | Menswear
  0790820001 -> Mr Tom padded jacket | Jacket | Black | Menswear


## AI product search (retrieval + Gemini)

This lets someone type what they want in plain words. It works in two steps. First I turn the request into a vector with the same sentence-transformer I used for products, and pull the 20 closest products from the 35,576 items that sold in the last 8 weeks. I keep only one colour per product, because otherwise the list fills up with the same item in different colours. Then I send the request and those 20 products to Gemini, with strict instructions to recommend only from that list, drop partial matches, and say so if nothing fits. It replies in JSON so I can check its answer automatically.

Retrieval on its own was good at the type of item but bad at hard rules. For "warm winter jacket for men", only 2 of the top 5 were actually men's jackets. The rest were kids' or sportswear. Gemini fixed that: from the 20 candidates it picked three men's jackets and nothing else.

Gemini's servers were overloaded during my first tries (error 503), and one request hung for over 3 minutes. So I added a 60-second timeout, automatic retries with growing waits, and a fallback to a lighter Gemini model. That's the version I'll use in the app.